In [6]:
## RNN Model Building LSTM (Long Short-Term Memory) is a type of recurrent neural network (RNN) architecture that

In [7]:
import sys
sys.path.insert(0, r"D:\machinelearning project")
import edf

KeyboardInterrupt: 

In [10]:
# Prepare the data for LSTM model
demand_col = df.filter(['demand'])           # Extract demand column (rename if needed e.g. 'load', 'consumption', 'mw')
dataset = demand_col.values                  # Convert to NumPy array
training_data_len = int(np.ceil(len(dataset) * 0.95)) # taking 50% of data as training data for LSTM model

NameError: name 'df' is not defined

In [11]:
# Preprocessing stages
df = df.dropna()
scaler = StandardScaler()                     # StandardScaler is used to standardize the features by removing the mean and scaling to unit variance
scaled_data = scaler.fit_transform(df)

training_data = scaled_data[0:training_data_len, 1:2]      # taking 50% of data as training data for LSTM model
X_train= []
Y_train = []

NameError: name 'df' is not defined

In [ ]:
# Create a sliding window of 24 hours to predict the next hour's demand
for i in range(24, len(training_data)):
    X_train.append(training_data[i-24:i, 0])  # past 24 hours of demand as features
    Y_train.append(training_data[i, 0])        # next hour's demand as target

X_train, Y_train = np.array(X_train), np.array(Y_train)

X_train = np.reshape(X_train, (X_train.shape[0], X_train.shape[1], 1)) # Reshaping the data to fit the LSTM model input requirements

In [ ]:
# Building the LSTM model
model = keras.models.Sequential()

#first layer
model.add(keras.layers.LSTM(64, return_sequences=True, input_shape=(X_train.shape[1], 1)))

#2ND LAYER
model.add(keras.layers.LSTM(64, return_sequences=False))

#3RD LAYER
model.add(keras.layers.Dense(128, activation='relu'))

#4TH LAYER
model.add(keras.layers.Dense(0.5))

#final output layer
model.add(keras.layers.Dense(1))

model.summary()
model.compile(optimizer='adam', loss='mae', metrics=[keras.metrics.RootMeanSquaredError()])

training = model.fit(X_train, Y_train, epochs=20, batch_size=32, validation_split=0.2)


In [ ]:
# Prepare the test data for LSTM model
test_data = scaled_data[training_data_len - 24:, :] # taking the last 24 hours of training data and all of the test data
X_test = []
Y_test = dataset[training_data_len:, :] # taking the test target variable data


for i in range(24, len(test_data)):
    X_test.append(test_data[i-24:i, 0])
    Y_test.append(test_data[i, 0])

X_test, Y_test = np.array(X_test), np.array(Y_test)
X_test = np.reshape(X_test, (X_test.shape[0], X_test.shape[1], 1))

In [ ]:
# Make predictions using the trained LSTM model
predictions = model.predict(X_test)
predictions = scaler.inverse_transform(predictions) # Inverse transform the predictions to get the original scale
Y_test = scaler.inverse_transform(Y_test.reshape(-1, 1)) # Inverse transform the test target variable to get the original scale   




# Evaluating the LSTM model using Mean Absolute Error and Root Mean Squared Error
mae_lstm = mean_absolute_error(Y_test, predictions) 
rmse_lstm = np.sqrt(mean_squared_error(Y_test, predictions))
print('LSTM RMSE:', rmse_lstm)
print('LSTM MAE:', mae_lstm)


In [ ]:
# Make predictions using the trained LSTM model
predictions = model.predict(X_test)
predictions = scaler.inverse_transform(predictions) # Inverse transform the predictions to get the original scale
Y_test = scaler.inverse_transform(Y_test.reshape(-1, 1)) # Inverse transform the test target variable to get the original scale   




# Evaluating the LSTM model using Mean Absolute Error and Root Mean Squared Error
mae_lstm = mean_absolute_error(Y_test, predictions) 
rmse_lstm = np.sqrt(mean_squared_error(Y_test, predictions))
print('LSTM RMSE:', rmse_lstm)
print('LSTM MAE:', mae_lstm)
